# rfe_sample_afm_daily_status.parquet 생성

In [ ]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\train_raw.parquet와
# C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_diff.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_afm_daily_status.parquet 생성

"""
Feature Name	Formula / Logic	Description
cascading_failure_flag	$\mathbb{I}(\Delta s197_t > 0 \land \Delta Timeout\_Total_t > 0)$	결함 도미노 연쇄 발생 플래그
data_corruption_hazard	$\mathbb{I}(\Delta s184_t > 0 \lor \Delta s199_t > 0)$	논리적 데이터 오염 위험 경고 플래그
is_warmup_7d	$\mathbb{I}(t_{elapsed} < 7)$	7일 관측 구간 부족 플래그
is_warmup_14d	$\mathbb{I}(t_{elapsed} < 14)$	14일 관측 구간 부족 플래그
is_warmup_28d	$\mathbb{I}(t_{elapsed} < 28)$	28일 관측 구간 부족 플래그
recovery_failure_flag	$\mathbb{I}(\Delta s197_t < 0) \cdot \mathbb{I}(\Delta s5_t > 0)$	대기 섹터의 불가 섹터 전이 현상
s184_1d_crash_flag	$\mathbb{I}(\Delta s184_t > 0)$	치명적 패리티 오류 1일 발생 플래그
s187_damaged	$\mathbb{I}(s187_t > 0)$	복구 불가 오류 손상 여부
s187_days_since_first	$\begin{cases} t - \min\{\tau \mid s187_\tau > 0\}, & \text{if } \exists \tau \\ -1, & \text{otherwise} \end{cases}$	smart_187 최초 발생 후 경과일
s187_ever_flag	$\mathbb{I}(\max_{0 \le i \le t}(s187_i) > 0)$	복구 불가 오류 횟수 누적 발생 플래그
s191_days_since_last	$\begin{cases} t - \max\{\tau \mid \max(0, \Delta s191_\tau) > 0\}, & \text{if } \exists \tau \\ -1, & \text{otherwise} \end{cases}$	외부 충격 발생 이후 무사고 경과일
s197_damaged	$\mathbb{I}(s197_t > 0)$	불안정 섹터 손상 여부
s197_recovery_flag	$\mathbb{I}(s197_t < s197_{t-1})$	불안정 섹터 회복 이벤트 플래그
s198_damaged	$\mathbb{I}(s198_t > 0)$	복구 불가 섹터 손상 여부
s199_days_since_last	$\begin{cases} t - \max\{\tau \mid \max(0, \Delta s199_\tau) > 0\}, & \text{if } \exists \tau \\ -1, & \text{otherwise} \end{cases}$	통신 오류 증가 이후 무사고 경과일
s5_damaged	$\mathbb{I}(s5_t > 0)$	불량 섹터 손상 여부
s5_days_since_first	$\begin{cases} t - \min\{\tau \mid s5_\tau > 0\}, & \text{if } \exists \tau \\ -1, & \text{otherwise} \end{cases}$	smart_5 최초 발생 후 경과일
s5_ever_flag	$\mathbb{I}(\max_{0 \le i \le t}(s5_i) > 0)$	불량 섹터 누적 발생 플래그
seek_damaged	$\mathbb{I}(Seek\_Error\_Count_t > 0)$	탐색 오류 손상 여부
timeout_5s_damaged	$\mathbb{I}(Timeout\_5s_t > 0)$	5초 이상 지연 손상 여부
timeout_total_days_since_last	$\begin{cases} t - \max\{\tau \mid \max(0, \Delta Timeout\_Total_\tau) > 0\}, & \text{if } \exists \tau \\ -1, & \text{otherwise} \end{cases}$	마지막 Timeout 증가 이후 경과일
zero_to_hero_count	$\sum_{k \in \mathcal{K}} \mathbb{I}(s_{k,t} > 0 \land s_{k,t-1} = 0)$	0이었다가 갑자기 튄 지표의 개수
"""

import duckdb
import os

BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
RAW_FILE = os.path.join(BASE_DIR, "data", "split_group_stratified", "train_raw.parquet").replace("\\", "/")
DIFF_FILE = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_diff.parquet").replace("\\", "/")
OUT_FILE = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_afm_daily_status.parquet").replace("\\", "/")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

con = duckdb.connect()

sql = f"""
COPY (
WITH base_raw AS (
    SELECT
        serial_number, date,
        smart_5_raw, smart_187_raw, smart_197_raw, smart_198_raw, smart_9_raw, smart_191_raw,
        seek_error_count, timeout_5s, timeout_total, Total_Seeks
    FROM read_parquet('{RAW_FILE}')
),

base_diff AS (
    SELECT
        serial_number, date,
        s5_diff, s184_diff, s187_diff, s191_diff, s197_diff, s199_diff,
        timeout_total_diff, s198_diff, seek_error_count_diff, timeout_5s_diff
    FROM read_parquet('{DIFF_FILE}')
),

joined AS (
    SELECT
        r.*,
        d.s5_diff, d.s184_diff, d.s187_diff, d.s191_diff,
        d.s197_diff, d.s199_diff, d.timeout_total_diff,
        d.s198_diff, d.seek_error_count_diff, d.timeout_5s_diff
    FROM base_raw r
    JOIN base_diff d
      ON r.serial_number = d.serial_number AND r.date = d.date
),

prep AS (
    SELECT
        *,
        CAST(date AS DATE) AS dt,

        LAG(smart_5_raw) OVER w AS s5_p,
        LAG(smart_187_raw) OVER w AS s187_p,
        LAG(smart_197_raw) OVER w AS s197_p,
        LAG(smart_198_raw) OVER w AS s198_p,
        LAG(seek_error_count) OVER w AS seek_p,
        LAG(timeout_5s) OVER w AS t5s_p

    FROM joined
    WINDOW w AS (PARTITION BY serial_number ORDER BY date)
),

calc AS (
    SELECT
        *,

        ROW_NUMBER() OVER w AS elapsed_days,

        MAX(smart_5_raw) OVER w AS s5_max_so_far,
        MAX(smart_187_raw) OVER w AS s187_max_so_far,

        -- ✅ FIXED: leakage-safe cumulative window (ORDER BY 포함)
        MIN(CASE WHEN smart_5_raw > 0 THEN dt END) OVER w AS s5_first_dt,
        MIN(CASE WHEN smart_187_raw > 0 THEN dt END) OVER w AS s187_first_dt,

        MAX(CASE WHEN s191_diff > 0 THEN dt END) OVER w AS s191_last_dt,
        MAX(CASE WHEN s199_diff > 0 THEN dt END) OVER w AS s199_last_dt,
        MAX(CASE WHEN timeout_total_diff > 0 THEN dt END) OVER w AS t_total_last_dt

    FROM prep
    WINDOW w AS (PARTITION BY serial_number ORDER BY date)
)

SELECT
    serial_number,
    dt AS date,

    CAST(elapsed_days < 7  AS UTINYINT) AS is_warmup_7d,
    CAST(elapsed_days < 14 AS UTINYINT) AS is_warmup_14d,
    CAST(elapsed_days < 28 AS UTINYINT) AS is_warmup_28d,

    CAST(smart_5_raw > 0 AS UTINYINT) AS s5_damaged,
    CAST(smart_187_raw > 0 AS UTINYINT) AS s187_damaged,
    CAST(smart_197_raw > 0 AS UTINYINT) AS s197_damaged,
    CAST(smart_198_raw > 0 AS UTINYINT) AS s198_damaged,
    CAST(seek_error_count > 0 AS UTINYINT) AS seek_damaged,
    CAST(timeout_5s > 0 AS UTINYINT) AS timeout_5s_damaged,

    CAST(s5_max_so_far > 0 AS UTINYINT) AS s5_ever_flag,
    CAST(s187_max_so_far > 0 AS UTINYINT) AS s187_ever_flag,

    CAST(s197_diff > 0 AND timeout_total_diff > 0 AS UTINYINT) AS cascading_failure_flag,
    CAST(s184_diff > 0 OR s199_diff > 0 AS UTINYINT) AS data_corruption_hazard,
    CAST(s197_diff < 0 AND s5_diff > 0 AS UTINYINT) AS recovery_failure_flag,
    CAST(s184_diff > 0 AS UTINYINT) AS s184_1d_crash_flag,
    CAST(s197_diff < 0 AS UTINYINT) AS s197_recovery_flag,

    CASE WHEN s5_first_dt IS NULL THEN -1
         ELSE CAST(date_diff('day', CAST(s5_first_dt AS DATE), dt) AS INTEGER)
    END AS s5_days_since_first,

    CASE WHEN s187_first_dt IS NULL THEN -1
         ELSE CAST(date_diff('day', CAST(s187_first_dt AS DATE), dt) AS INTEGER)
    END AS s187_days_since_first,

    CASE WHEN s191_last_dt IS NULL THEN -1
         ELSE CAST(date_diff('day', CAST(s191_last_dt AS DATE), dt) AS INTEGER)
    END AS s191_days_since_last,

    CASE WHEN s199_last_dt IS NULL THEN -1
         ELSE CAST(date_diff('day', CAST(s199_last_dt AS DATE), dt) AS INTEGER)
    END AS s199_days_since_last,

    CASE WHEN t_total_last_dt IS NULL THEN -1
         ELSE CAST(date_diff('day', CAST(t_total_last_dt AS DATE), dt) AS INTEGER)
    END AS timeout_total_days_since_last,

    CAST(
        (CASE WHEN smart_5_raw > 0 AND s5_p = 0 THEN 1 ELSE 0 END) +
        (CASE WHEN smart_187_raw > 0 AND s187_p = 0 THEN 1 ELSE 0 END) +
        (CASE WHEN smart_197_raw > 0 AND s197_p = 0 THEN 1 ELSE 0 END) +
        (CASE WHEN smart_198_raw > 0 AND s198_p = 0 THEN 1 ELSE 0 END) +
        (CASE WHEN seek_error_count > 0 AND seek_p = 0 THEN 1 ELSE 0 END) +
        (CASE WHEN timeout_5s > 0 AND t5s_p = 0 THEN 1 ELSE 0 END)
    AS UTINYINT) AS zero_to_hero_count

FROM calc
) TO '{OUT_FILE}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""

con.execute(sql)
con.close()

print("Daily Status Feature Extraction Complete (Leakage-fixed)")

# rfe_sample_afm_daily_impact.parquet 생성

In [ ]:
# # C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\train_raw.parquet와
# # C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_diff.parquet으로부터
# # C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_afm_daily_impact.parquet 생성

"""
Feature Name	Formula / Logic	Description
age_weighted_seek_error	$\Delta Seek\_Error_t \times s9_t$	노후도 대비 탐색 오류 심각도
age_weighted_workload	$\ln(\lvert\Delta s241_t + \Delta s242_t\rvert + 1) \times \ln(s9_t + 1)$	노후도 기반 일일 작업 부하 가중치
cumulative_error_score	$5\lvert\Delta s5_t\rvert + 4\lvert\Delta s187_t\rvert + 3\lvert\Delta Timeout\_Total_t\rvert + 2\lvert\Delta s197_t\rvert + 1\lvert\Delta s198_t\rvert$	핵심 5대 변수 종합 위험 점수
error_growth_ratio	$\frac{E_t + 1}{E_{t-1} + 1}$	오류 가속화 비율
error_saturation_score	$\mathbb{I}(s187_t \ge 65535) + \mathbb{I}(s189_t \ge 65535) + \mathbb{I}(Seek\_Error_t \ge 65535)$	핵심 오류 지표 포화도 점수
fatal_crash_interaction	$\Delta Timeout\_5s_t \times \Delta s187_t$	응답 지연 및 복구 불가 오류 결합 지수
firmware_struggle_index	$(\Delta s197_t + \Delta s198_t) \times \Delta Timeout\_Total_t$	펌웨어 오류 복구 지연(발악) 지수
io_asymmetry_index	$\frac{\lvert\Delta s241_t - \Delta s242_t\rvert}{\Delta s241_t + \Delta s242_t + 1}$	읽기/쓰기 작업 비대칭 지수
late_stage_degradation	$\max(0, \Delta s5_t) \times s9_t$	노후화 말기 불량 섹터 발생 위험도
log_shock_fly_interaction	$\ln(1 + \lvert\Delta s191_t \times \Delta s189_t\rvert) \times \text{sgn}(\Delta s191_t \times \Delta s189_t)$	외부 충격 및 헤드 비행 오류 동시 발생 심각도
multi_error_coincidence	$\sum_{k} \mathbb{I}(\Delta s_{k,t} > 0)$	다중 오류 동시 발생 지표
pending_to_offline_ratio	$\frac{s197_t + 1}{s198_t + 1}$	대기 섹터의 복구 불가 전이 심각도
reallocated_pending_ratio	$\frac{s197_t + 1}{s5_t + 1}$	불량 섹터 대비 불안정 섹터 비율
s5_daily_failure_speed	$\max(0, \Delta s5_t)$	일일 불량 섹터 노후화 속도
s187_error_rate	$\frac{s187_t}{s9_t + 1}$	시간당 복구 불가 오류 발생률
s198_error_rate	$\frac{s198_t}{s9_t + 1}$	시간당 복구 불가 섹터 발생률
s199_error_density	$\frac{s199_t}{Total\_Reads_t + Total\_Seeks_t + 1}$	작업당 통신 연결 오류 밀도
seek_error_density	$\frac{Seek\_Error\_Count_t}{Total\_Seeks_t + 1}$	탐색당 탐색 오류 밀도
shock_fatigue_rate	$\frac{s191_t}{s9_t + 1}$	시간당 외부 충격 누적 피로도
shock_seek_interaction	$\Delta s191_t \times \Delta Seek\_Error_t$	충격 및 탐색 오류 동시 발생 결합 지수
temp_error_index	$\max(0, s194_t - 40) \times E_t$	고온 환경 에러 발생 지수
thermal_stress_index	$\max(0, s194_t - 40)^2 \times \Delta s241_t$	고온 스트레스 및 누적 쓰기량 결합 지수
timeout_read_density	$\frac{Timeout\_Total_t}{Total\_Reads_t + 1}$	읽기 작업 대비 에러율 (밀도)
timeout_seek_density	$\frac{Timeout\_Total_t}{Total\_Seeks_t + 1}$	탐색 작업 대비 에러율 (밀도)
timeout_severity_score	$\frac{Timeout\_Total_t + 1}{Timeout\_5s_t + 1}$	타임아웃 심각도 비율
timeout_to_uncorrectable_lag1	$\max(0, \Delta Timeout\_Total_t) \times \max(0, \Delta s198_{t-1})$	시차 결합 지수
workload_intensity	$\frac{s9_t + 1}{\Delta s241_t + \Delta s242_t + 1}$	누적 사용 시간 대비 총 작업량 비율
write_stability_ratio	$\frac{s189_t + 1}{\Delta s241_t + 1}$	쓰기량 대비 헤드 정렬 불량률
"""

import duckdb
import os

BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
RAW_FILE = os.path.join(BASE_DIR, "data", "split_group_stratified", "train_raw.parquet").replace("\\", "/")
DIFF_FILE = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_diff.parquet").replace("\\", "/")
OUT_FILE = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_afm_daily_impact.parquet").replace("\\", "/")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

con = duckdb.connect()

sql = f"""
COPY (
    WITH base_raw AS (
        SELECT * FROM read_parquet('{RAW_FILE}')
    ),
    base_diff AS (
        SELECT * FROM read_parquet('{DIFF_FILE}')
    ),
    joined AS (
        SELECT
            r.*,
            d.s5_diff, d.s184_diff, d.s187_diff, d.s191_diff, d.s197_diff, d.s189_diff, d.s199_diff, 
            d.timeout_total_diff, d.s198_diff, d.seek_error_count_diff, d.timeout_5s_diff, d.s241_diff, d.s242_diff
        FROM base_raw r
        JOIN base_diff d ON r.serial_number = d.serial_number AND r.date = d.date
    ),
    prep AS (
        SELECT
            *,
            CAST(date AS DATE) AS dt,
            1.0 * (ABS(s5_diff) + ABS(s187_diff) + ABS(s197_diff) + ABS(s198_diff) + ABS(timeout_total_diff)) AS daily_errors,
            (smart_9_raw - LAG(smart_9_raw) OVER w0) AS s9_diff,
            LAG(1.0 * (ABS(s5_diff) + ABS(s187_diff) + ABS(s197_diff) + ABS(s198_diff) + ABS(timeout_total_diff))) OVER w0 AS prev_daily_errors,
            LAG(s198_diff) OVER w0 AS prev_s198_diff
        FROM joined
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    calc AS (
        SELECT
            *,
            seek_error_count_diff * smart_9_raw AS age_weighted_seek_error,
            timeout_5s_diff * s187_diff AS fatal_crash_interaction,
            s191_diff * seek_error_count_diff AS shock_seek_interaction,
            -- Interactive & Severity
            ln(ABS(s241_diff + s242_diff) + 1.0) * ln(smart_9_raw + 1.0) AS age_weighted_workload,
            GREATEST(0, s5_diff) * smart_9_raw AS late_stage_degradation,
            (5*ABS(s5_diff) + 4*ABS(s187_diff) + 3*ABS(timeout_total_diff) + 2*ABS(s197_diff) + 1*ABS(s198_diff)) AS cumulative_error_score,
            (s197_diff + s198_diff) * timeout_total_diff AS firmware_struggle_index,
            SIGN(s191_diff * s189_diff) * ln(1.0 + ABS(s191_diff * s189_diff)) AS log_shock_fly_interaction,
            
            (daily_errors + 1.0) / (COALESCE(prev_daily_errors, 0) + 1.0) AS error_growth_ratio,
            ABS(s241_diff - s242_diff) / (ABS(s241_diff + s242_diff) + 1.0) AS io_asymmetry_index,
            
            (smart_197_raw + 1.0) / (smart_198_raw + 1.0) AS pending_to_offline_ratio,
            (smart_197_raw + 1.0) / (smart_5_raw + 1.0) AS reallocated_pending_ratio,
            GREATEST(0, s5_diff) AS s5_daily_failure_speed,
            
            smart_187_raw / (smart_9_raw + 1.0) AS s187_error_rate,
            smart_198_raw / (smart_9_raw + 1.0) AS s198_error_rate,
            smart_199_raw / (Total_Reads + Total_Seeks + 1.0) AS s199_error_density,
            seek_error_count / (Total_Seeks + 1.0) AS seek_error_density,
            GREATEST(0, s191_diff) / (GREATEST(0, s9_diff) + 1.0) AS shock_fatigue_rate,
            
            timeout_total / (Total_Reads + 1.0) AS timeout_read_density,
            timeout_total / (Total_Seeks + 1.0) AS timeout_seek_density,
            LN(1 + GREATEST(timeout_total_diff, 0)) - LN(1 + GREATEST(timeout_5s_diff, 0)) AS timeout_severity_score,
            
            (smart_241_raw + smart_242_raw + 1.0) / (smart_9_raw + 1.0) AS workload_intensity,
            (smart_189_raw + 1.0) / (smart_241_raw + 1.0) AS write_stability_ratio,
            
            GREATEST(0, smart_194_raw - 40) * daily_errors AS temp_error_index,
            POWER(GREATEST(0, smart_194_raw - 40), 2) * s241_diff AS thermal_stress_index,
            
            (CAST(smart_187_raw >= 65535 AS INT) + CAST(smart_189_raw >= 65535 AS INT) + CAST(seek_error_count >= 65535 AS INT)) AS error_saturation_score,
            (CAST(s5_diff > 0 AS INT) + CAST(s187_diff > 0 AS INT) + CAST(s197_diff > 0 AS INT) + CAST(s198_diff > 0 AS INT) + CAST(timeout_total_diff > 0 AS INT)) AS multi_error_count,
            GREATEST(0, timeout_total_diff) * COALESCE(GREATEST(0, prev_s198_diff), 0) AS timeout_to_uncorrectable_lag1
        FROM prep
    )
    SELECT
        serial_number, dt AS date,
        CAST(COALESCE(age_weighted_seek_error, 0) AS FLOAT) AS age_weighted_seek_error,
        CAST(COALESCE(fatal_crash_interaction, 0) AS FLOAT) AS fatal_crash_interaction,
        CAST(COALESCE(shock_seek_interaction, 0) AS FLOAT) AS shock_seek_interaction,
        CAST(COALESCE(age_weighted_workload, 0) AS FLOAT) AS age_weighted_workload,
        CAST(COALESCE(late_stage_degradation, 0) AS FLOAT) AS late_stage_degradation,
        CAST(COALESCE(cumulative_error_score, 0) AS FLOAT) AS cumulative_error_score,
        CAST(COALESCE(firmware_struggle_index, 0) AS FLOAT) AS firmware_struggle_index,
        CAST(COALESCE(log_shock_fly_interaction, 0) AS FLOAT) AS log_shock_fly_interaction,
        CAST(COALESCE(error_growth_ratio, 0) AS FLOAT) AS error_growth_ratio,
        CAST(COALESCE(io_asymmetry_index, 0) AS FLOAT) AS io_asymmetry_index,
        CAST(COALESCE(pending_to_offline_ratio, 0) AS FLOAT) AS pending_to_offline_ratio,
        CAST(COALESCE(reallocated_pending_ratio, 0) AS FLOAT) AS reallocated_pending_ratio,
        CAST(COALESCE(s5_daily_failure_speed, 0) AS FLOAT) AS s5_daily_failure_speed,
        CAST(COALESCE(s187_error_rate, 0) AS FLOAT) AS s187_error_rate,
        CAST(COALESCE(s198_error_rate, 0) AS FLOAT) AS s198_error_rate,
        CAST(COALESCE(s199_error_density, 0) AS FLOAT) AS s199_error_density,
        CAST(COALESCE(seek_error_density, 0) AS FLOAT) AS seek_error_density,
        CAST(COALESCE(shock_fatigue_rate, 0) AS FLOAT) AS shock_fatigue_rate,
        CAST(COALESCE(timeout_read_density, 0) AS FLOAT) AS timeout_read_density,
        CAST(COALESCE(timeout_seek_density, 0) AS FLOAT) AS timeout_seek_density,
        CAST(COALESCE(timeout_severity_score, 0) AS FLOAT) AS timeout_severity_score,
        CAST(COALESCE(workload_intensity, 0) AS FLOAT) AS workload_intensity,
        CAST(COALESCE(write_stability_ratio, 0) AS FLOAT) AS write_stability_ratio,
        CAST(COALESCE(temp_error_index, 0) AS FLOAT) AS temp_error_index,
        CAST(COALESCE(thermal_stress_index, 0) AS FLOAT) AS thermal_stress_index,
        CAST(COALESCE(error_saturation_score, 0) AS FLOAT) AS error_saturation_score,
        CAST(COALESCE(multi_error_count, 0) AS FLOAT) AS multi_error_count,
        CAST(COALESCE(timeout_to_uncorrectable_lag1, 0) AS FLOAT) AS timeout_to_uncorrectable_lag1
    FROM calc
) TO '{OUT_FILE}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""

con.execute(sql)
con.close()
print("Daily Impact Feature Extraction Complete")



<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
C:\Users\joon6\AppData\Local\Temp\ipykernel_14560\835054956.py:5: SyntaxWarning: invalid escape sequence '\D'
  """


Daily Impact Feature Extraction Complete


# rfe_sample_afm_windowed.parquet 생성

In [ ]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\train_raw.parquet와
# C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_diff.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_afm_windowed.parquet 생성

"""
이 코드에서 만드는 파생변수 목록
error_density_14d	$\frac{\sum_{i=0}^{13} E_{t-i}}{\sum_{i=0}^{13} \max(0, \Delta s9_{t-i}) + 1}$	14일간 시간 대비 에러 밀도
read_spike_ratio	$\frac{\Delta s242_t}{\left( \frac{1}{7}\sum_{i=1}^{7} \Delta s242_{t-i} \right) + 1}$	7일 평균 대비 당일 읽기량 폭주 비율
s5_relative_score_14d	$\frac{s5_t}{P_{95}(s5_{t-14:t-1}) + 1}$	14일 최고점 대비 당일 불량 섹터 비율
s187_14d_burst_index	$\sum_{i=0}^{13} \left( \Delta s187_{t-i} \cdot \mathbb{I}(\Delta s187_{t-i} > 0) \right)$	복구 불가 오류 횟수 중기 누적 증가량
s189_28d_highfly_burst	$\sum_{i=0}^{27} \Delta s189_{t-i}$	28일간 불량 쓰기 증가량
s192_14d_burst	$\sum_{i=0}^{13} \Delta s192_{t-i}$	14일 강제 종료 단기 폭주량
s194_over40_7d_count	$\sum_{i=0}^{6} \mathbb{I}(s194_{t-i} > 40)$	7일 중 온도가 40도를 초과한 일수
s197_7d_straight_rise	$\mathbb{I}\left( \sum_{i=0}^{6} \mathbb{I}(\Delta s197_{t-i} > 0) \ge 5 \right)$	7일 중 5일 이상 연속 상승 여부
s199_14d_burst	$\sum_{i=0}^{13} \Delta s199_{t-i}$	14일간 통신 오류 단기 폭주량
seek_error_14d_spike_ratio	$\frac{\max(0, \Delta Seek\Error_t)}{AVG{14d}(\max(0, \Delta Seek\Error_{t-i})) + 1}$	탐색 오류 14일 대비 이상치 비율
seek_spike_ratio	$\frac{\Delta Total\_Seeks_t}{\left( \frac{1}{7}\sum_{i=1}^{7} \Delta Total\_Seeks_{t-i} \right) + 1}$	7일 평균 대비 당일 탐색 폭주 비율
shock_to_highfly_ratio	$\frac{\sum_{i=0}^{27} \lvert\Delta s191_{t-i}\rvert + 1}{\sum_{i=0}^{27} \lvert\Delta s189_{t-i}\rvert + 1}$	충격 대비 헤드 불안정 전이 비율
thermal_fatigue_integral_7d	$\sum_{i=0}^{6} \max(0, s194_{t-i} - 40)$	7일간 40도 초과 열 피로 누적량(면적)
uncorrectable_spike_ratio	$\frac{\Delta s198_t}{\left( \frac{1}{14}\sum_{i=1}^{14} \Delta s198_{t-i} \right) + 1}$	복구 불가 섹터 중기 폭증 비율
workload_7d_accel	$(\Delta s241_t + \Delta s242_t) - (\Delta s241_{t-7} + \Delta s242_{t-7})$	총 작업량 7일 가속도
write_spike_ratio	$\frac{\Delta s241_t}{\left( \frac{1}{7}\sum_{i=1}^{7} \Delta s241_{t-i} \right) + 1}$	7일 평균 대비 당일 쓰기량 폭주 비율
"""

import duckdb
import os

BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
RAW_FILE = os.path.join(BASE_DIR, "data", "split_group_stratified", "train_raw.parquet").replace("\\", "/")
IN_FILE  = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_diff.parquet").replace("\\", "/")
OUT_FILE = os.path.join(BASE_DIR, "data", "rfe_sample_data", "rfe_sample_afm_windowed.parquet").replace("\\", "/")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

con = duckdb.connect()

print("1/1. DuckDB를 사용하여 전체 데이터를 한 번에 처리합니다...")


sql = f"""
COPY (
WITH base AS (
    SELECT 
        d.*,
        CAST(d.date AS DATE) AS dt,
        COALESCE(r.smart_5_raw, 0) AS smart_5_raw,
        COALESCE(r.smart_9_raw, 0) AS smart_9_raw,
        COALESCE(r.smart_241_raw, 0) AS smart_241_raw,
        COALESCE(r.smart_242_raw, 0) AS smart_242_raw,
        COALESCE(r.smart_187_raw, 0) AS smart_187_raw,
        COALESCE(r.smart_189_raw, 0) AS smart_189_raw,
        COALESCE(r.smart_192_raw, 0) AS smart_192_raw,
        COALESCE(r.smart_194_raw, 0) AS smart_194_raw,
        COALESCE(r.seek_error_count, 0) AS seek_error_count,
        COALESCE(r.Total_Seeks, 0) AS Total_Seeks
    FROM read_parquet('{IN_FILE}') d
    LEFT JOIN read_parquet('{RAW_FILE}') r
    USING (serial_number, date)
),

diffs AS (
    SELECT *,
        (COALESCE(s241_diff, 0) + COALESCE(s242_diff, 0)) AS daily_workload,
        (COALESCE(s5_diff, 0) + COALESCE(s187_diff, 0) + COALESCE(s197_diff, 0) + COALESCE(s198_diff, 0) + COALESCE(timeout_total_diff, 0)) AS total_errors_diff,
        (seek_error_count - LAG(seek_error_count) OVER w0) AS seek_diff,
        (Total_Seeks - LAG(Total_Seeks) OVER w0) AS total_seeks_diff,
        (smart_9_raw - LAG(smart_9_raw) OVER w0) AS s9_diff,
        (smart_192_raw - LAG(smart_192_raw) OVER w0) AS s192_diff
    FROM base
    WINDOW w0 AS (PARTITION BY serial_number ORDER BY dt)
),

windows AS (
    SELECT *,
        COALESCE(SUM(total_errors_diff) OVER w14, 0) AS sum_errors_14d,
        COALESCE(SUM(GREATEST(0, s9_diff)) OVER w14, 0) AS sum_s9_diff_14d,
        
        COALESCE(SUM(GREATEST(0, s187_diff)) OVER w14, 0) AS s187_14d_burst_index,
        COALESCE(SUM(s189_diff) OVER w28, 0) AS s189_28d_highfly_burst,
        COALESCE(SUM(s192_diff) OVER w14, 0) AS s192_14d_burst,
        COALESCE(SUM(s199_diff) OVER w14, 0) AS s199_14d_burst,
        
        COALESCE(AVG(GREATEST(0, s242_diff)) OVER w7_pre, 0) AS avg_read_diff_7d_pre,
        COALESCE(AVG(GREATEST(0, s241_diff)) OVER w7_pre, 0) AS avg_write_diff_7d_pre,
        COALESCE(AVG(GREATEST(0, s198_diff)) OVER w14_pre, 0) AS avg_uncorrectable_diff_14d_pre,
        COALESCE(AVG(GREATEST(0, seek_diff)) OVER w14_pre, 0) AS avg_seek_diff_14d_pre,
        COALESCE(AVG(GREATEST(0, total_seeks_diff)) OVER w7_pre, 0) AS avg_total_seeks_diff_7d_pre,
        
        COALESCE(SUM(CASE WHEN smart_194_raw > 40 THEN 1 ELSE 0 END) OVER w7, 0) AS s194_over40_7d_count,
        COALESCE(SUM(GREATEST(0, smart_194_raw - 40)) OVER w7, 0) AS thermal_fatigue_integral_7d,
        COALESCE(SUM(CASE WHEN s197_diff > 0 THEN 1 ELSE 0 END) OVER w7, 0) AS s197_rose_7d_count,
        
        -- s5 상위 95% 분위수 계산 (D-14 ~ D-1)
        COALESCE(quantile_cont(smart_5_raw, 0.95) OVER w14_pre, 0) AS p95_s5_14d_pre,
        
        COALESCE(SUM(ABS(s191_diff)) OVER w28, 0) AS sum_abs_s191_28d,
        COALESCE(SUM(ABS(s189_diff)) OVER w28, 0) AS sum_abs_s189_28d,
        
        COALESCE(LAG(daily_workload, 7) OVER w0, 0) AS daily_workload_lag7
        
    FROM diffs
    WINDOW
        w0 AS (PARTITION BY serial_number ORDER BY dt),
        w7 AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),
        w7_pre AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING),
        w14 AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
        w14_pre AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING),
        w28 AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 27 PRECEDING AND CURRENT ROW)
),

calc AS (
    SELECT 
        serial_number, date,

        sum_errors_14d / (sum_s9_diff_14d + 1.0) AS error_density_14d,
        s187_14d_burst_index,
        s189_28d_highfly_burst,
        s192_14d_burst,
        s199_14d_burst,

        GREATEST(0, COALESCE(s242_diff, 0)) / (avg_read_diff_7d_pre + 1.0) AS read_spike_ratio,
        GREATEST(0, COALESCE(s241_diff, 0)) / (avg_write_diff_7d_pre + 1.0) AS write_spike_ratio,
        GREATEST(0, COALESCE(s198_diff, 0)) / (avg_uncorrectable_diff_14d_pre + 1.0) AS uncorrectable_spike_ratio,
        GREATEST(0, COALESCE(seek_diff, 0)) / (avg_seek_diff_14d_pre + 1.0) AS seek_error_14d_spike_ratio,
        GREATEST(0, COALESCE(total_seeks_diff, 0)) / (avg_total_seeks_diff_7d_pre + 1.0) AS seek_spike_ratio,

        s194_over40_7d_count,
        thermal_fatigue_integral_7d,
        CAST(CASE WHEN s197_rose_7d_count >= 5 THEN 1 ELSE 0 END AS UTINYINT) AS s197_7d_straight_rise,

        -- 기존 relative_score
        smart_5_raw / (p95_s5_14d_pre + 1.0) AS s5_relative_score_14d,

        (sum_abs_s191_28d + 1.0) / (sum_abs_s189_28d + 1.0) AS shock_to_highfly_ratio,
        daily_workload - daily_workload_lag7 AS workload_7d_accel
    FROM windows
)

SELECT
    serial_number, date,
    CAST(COALESCE(error_density_14d, 0) AS FLOAT) AS error_density_14d,
    CAST(COALESCE(read_spike_ratio, 0) AS FLOAT) AS read_spike_ratio,
    CAST(COALESCE(s5_relative_score_14d, 0) AS FLOAT) AS s5_relative_score_14d,
    CAST(COALESCE(s187_14d_burst_index, 0) AS FLOAT) AS s187_14d_burst_index,
    CAST(COALESCE(s189_28d_highfly_burst, 0) AS FLOAT) AS s189_28d_highfly_burst,
    CAST(COALESCE(s192_14d_burst, 0) AS FLOAT) AS s192_14d_burst,
    CAST(COALESCE(s194_over40_7d_count, 0) AS FLOAT) AS s194_over40_7d_count,
    CAST(COALESCE(s197_7d_straight_rise, 0) AS FLOAT) AS s197_7d_straight_rise,
    CAST(COALESCE(s199_14d_burst, 0) AS FLOAT) AS s199_14d_burst,
    CAST(COALESCE(seek_error_14d_spike_ratio, 0) AS FLOAT) AS seek_error_14d_spike_ratio,
    CAST(COALESCE(seek_spike_ratio, 0) AS FLOAT) AS seek_spike_ratio,
    CAST(COALESCE(shock_to_highfly_ratio, 0) AS FLOAT) AS shock_to_highfly_ratio,
    CAST(COALESCE(thermal_fatigue_integral_7d, 0) AS FLOAT) AS thermal_fatigue_integral_7d,
    CAST(COALESCE(uncorrectable_spike_ratio, 0) AS FLOAT) AS uncorrectable_spike_ratio,
    CAST(COALESCE(workload_7d_accel, 0) AS FLOAT) AS workload_7d_accel,
    CAST(COALESCE(write_spike_ratio, 0) AS FLOAT) AS write_spike_ratio
FROM calc
) TO '{OUT_FILE}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""
con.execute(sql)
con.close()
print(f"🎉 AFM 변수 생성 완료! (저장: {OUT_FILE})")


<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\joon6\AppData\Local\Temp\ipykernel_12664\518159993.py:5: SyntaxWarning: invalid escape sequence '\s'
  """


1/1. DuckDB를 사용하여 전체 데이터를 한 번에 처리합니다...
🎉 AFM 변수 생성 완료! (저장: C:/Workspace/06_ML_projdect/26_1_COIN/data/rfe_sample_data/rfe_sample_afm_windowed.parquet)
